<a href="https://colab.research.google.com/github/noname123920/diplom/blob/main/pinn_deepxde.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [52]:
!pip install deepxde

In [53]:
import os
os.environ["DDE_BACKEND"] = "pytorch"
import deepxde as dde
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import torch
import pandas as pd

In [54]:
kappa = 1.0
geom = dde.geometry.Interval(0, np.pi)
timedomain = dde.geometry.TimeDomain(0, 1)
geomtime = dde.geometry.GeometryXTime(geom, timedomain)

In [55]:
def pde(x, p):
    dp_t = dde.grad.jacobian(p, x, i=0, j=1)
    dp_xx = dde.grad.hessian(p, x, i=0, j=0)
    return dp_t - kappa * dp_xx

In [56]:
data = dde.data.TimePDE(
    geomtime,
    pde,
    [],
    num_domain=114,
    train_distribution="uniform"
)

In [57]:
layer_size = [2] + [20] * 2 + [1]
activation = "tanh"
initializer = "Glorot uniform"
net = dde.nn.FNN(layer_size, activation, initializer)

In [58]:
def output_transform(x, y):
    x_col = x[:, 0:1]
    t_col = x[:, 1:2]
    return torch.sin(x_col) + x_col * (np.pi - x_col) * t_col * y

In [59]:
net.apply_output_transform(output_transform)

In [60]:
model = dde.Model(data, net)
model.compile("adam", lr=1e-3)
losshistory, train_state = model.train(iterations=1000, display_every=50)

Compiling model...
'compile' took 0.000225 s

Training model...

Step      Train loss    Test loss     Test metric
0         [1.41e+01]    [1.41e+01]    []  
50        [8.64e-02]    [8.64e-02]    []  
100       [4.12e-02]    [4.12e-02]    []  
150       [2.72e-02]    [2.72e-02]    []  
200       [1.81e-02]    [1.81e-02]    []  
250       [1.27e-02]    [1.27e-02]    []  
300       [9.66e-03]    [9.66e-03]    []  
350       [7.96e-03]    [7.96e-03]    []  
400       [6.88e-03]    [6.88e-03]    []  
450       [6.09e-03]    [6.09e-03]    []  
500       [5.44e-03]    [5.44e-03]    []  
550       [4.87e-03]    [4.87e-03]    []  
600       [4.37e-03]    [4.37e-03]    []  
650       [3.92e-03]    [3.92e-03]    []  
700       [3.51e-03]    [3.51e-03]    []  
750       [3.16e-03]    [3.16e-03]    []  
800       [2.84e-03]    [2.84e-03]    []  
850       [2.56e-03]    [2.56e-03]    []  
900       [2.31e-03]    [2.31e-03]    []  
950       [2.09e-03]    [2.09e-03]    []  
1000      [1.90e-03]    [

In [61]:
points = data.train_x
fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, np.pi, np.pi, 0, 0], y=[0, 0, 1, 1, 0], mode="lines", fill="toself", fillcolor="rgba(0, 0, 255, 0.05)"))
fig.add_trace(go.Scatter(x=points[:, 0], y=points[:, 1], mode="markers"))
fig.update_layout(xaxis_title="x", yaxis_title="t", showlegend=False)
fig.show()

In [62]:
X, T = np.meshgrid(np.linspace(0, np.pi, 100), np.linspace(0, 1, 100))
inputs = np.column_stack([X.ravel(), T.ravel()])

$$
E_{L_2} = \frac{\| p_{\text{exact}} - p_{\text{pred}} \|_2}{\| p_{\text{exact}} \|_2} \times 100\% = \frac{\sqrt{\sum_{i=1}^{N} (p_{\text{exact}, i} - p_{\text{pred}, i})^2}}{\sqrt{\sum_{i=1}^{N} p_{\text{exact}, i}^2}} \times 100\%
$$

где $N = 10\,000$ — количество узлов равномерной тестовой сетки ($100 \times 100$).


In [63]:
p_pred = model.predict(inputs).ravel()
p_exact = (np.exp(-T) * np.sin(X)).ravel()
l2_error = (np.linalg.norm(p_pred - p_exact) / np.linalg.norm(p_exact)) * 100
l2_error

np.float64(1.5606175892502672)

In [64]:
fig = px.line(
    x=X.ravel(),
    y=[model.predict(inputs).ravel(), (np.exp(-T) * np.sin(X)).ravel()],
    animation_frame=np.round(T.ravel(), 3),
    range_y=[-0.05, 1.05],
    labels={'x': 'x', 'value': 'p(x, t)', 'animation_frame': 'Time'}
)
fig.for_each_trace(lambda t: t.update(name='PINN' if t.name == 'wide_variable_0' else 'Exact'))
fig.show()

## Что будет если менять число эпох?

In [67]:
X, T = np.meshgrid(np.linspace(0, np.pi, 100), np.linspace(0, 1, 100))
inputs = np.column_stack([X.ravel(), T.ravel()])

In [73]:
results = {}

for iters in (2000, 3000, 4000):
  models, histories, states = [], [], []
  for i in range(5):
    net = dde.nn.FNN(layer_size, activation, initializer)
    net.apply_output_transform(output_transform)
    model = dde.Model(data, net)
    model.compile("adam", lr=1e-3)
    losshistory, train_state = model.train(iterations=iters)

    models.append(model)
    histories.append(losshistory)
    states.append(train_state)

  results[iters] = {"models": models, "histories": histories, "states": states}

Compiling model...
'compile' took 0.000313 s

Training model...

Step      Train loss    Test loss     Test metric
0         [3.08e+00]    [3.08e+00]    []  
1000      [2.09e-04]    [2.09e-04]    []  
2000      [1.07e-04]    [1.07e-04]    []  

Best model at step 2000:
  train loss: 1.07e-04
  test loss: 1.07e-04
  test metric: []

'train' took 16.254426 s

Compiling model...
'compile' took 0.000197 s

Training model...

Step      Train loss    Test loss     Test metric
0         [4.06e+00]    [4.06e+00]    []  
1000      [5.70e-04]    [5.70e-04]    []  
2000      [1.50e-04]    [1.50e-04]    []  

Best model at step 2000:
  train loss: 1.50e-04
  test loss: 1.50e-04
  test metric: []

'train' took 16.859337 s

Compiling model...
'compile' took 0.000367 s

Training model...

Step      Train loss    Test loss     Test metric
0         [1.12e+01]    [1.12e+01]    []  
1000      [4.31e-04]    [4.31e-04]    []  
2000      [1.49e-04]    [1.49e-04]    []  

Best model at step 2000:
  train lo

In [75]:
p_exact = (np.exp(-T) * np.sin(X)).ravel()

for iters, data_dict in results.items():
  errs = [
      float(
          np.linalg.norm(m.predict(inputs).ravel() - p_exact)
          / np.linalg.norm(p_exact)
          * 100
      )
      for m in data_dict["models"]
  ]
  data_dict["l2_errors"] = errs

  errs_str = ", ".join(f"{e:.4f}%" for e in errs)
  print(f"{iters} итер. | Ср: {np.mean(errs):.4f}% ± {np.std(errs):.4f}%")
  print(f"   Модели: {errs_str}\n")


2000 итер. | Ср: 0.5436% ± 0.2135%
   Модели: 0.3081%, 0.7114%, 0.7825%, 0.6487%, 0.2675%

3000 итер. | Ср: 0.3137% ± 0.1372%
   Модели: 0.4387%, 0.2094%, 0.1256%, 0.4928%, 0.3019%

4000 итер. | Ср: 0.2556% ± 0.0723%
   Модели: 0.1864%, 0.3684%, 0.1888%, 0.2233%, 0.3112%

